In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from scipy.ndimage import rotate
from skimage.filters import threshold_multiotsu
from skimage.transform import downscale_local_mean

# --- CONFIGURACIÓN ---
BASE_PATH = r"C:\Users\marta\Downloads\workdir"
INPUT_DIR = os.path.join(BASE_PATH, "raw/negatives") # Usaremos sujetos reales para la prueba
OUTPUT_DIR = os.path.join(BASE_PATH, "synthetic_pilot")
os.makedirs(os.path.join(OUTPUT_DIR, "imagesTr"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "labelsTr"), exist_ok=True)

def generate_sCMB_momeni(vol_mm3, voxel_size, strength=0.9):
    """Implementa la generación 3D Gaussiana de Momeni (v1)."""
    # Constantes del artículo
    K = 1.175 
    sigma_t = np.cbrt((3 * vol_mm3) / (4 * np.pi * (K**3)))
    
    # Variación de esfericidad
    rmin, rmax = 0.5, 0.9
    sx_mm = sigma_t * (np.random.uniform(rmin, rmax))
    sy_mm = sigma_t * (np.random.uniform(rmin, rmax))
    # Cálculo de sz para mantener el volumen constante
    sz_mm = (sigma_t**3) / (sx_mm * sy_mm)
    
    # Oversampling para ADNI (1x1x4) -> Rejilla física en mm
    # Definimos un cubo de 10mm para generar la lesión
    res = 0.1 # mm
    grid_lim = 5.0 # mm
    x = np.arange(-grid_lim, grid_lim, res)
    y = np.arange(-grid_lim, grid_lim, res)
    z = np.arange(-grid_lim, grid_lim, res)
    xx, yy, zz = np.meshgrid(x, y, z, indexing='ij')
    
    # Fórmula Gaussiana Momeni
    exponent = -((xx**2)/(2*sx_mm**2) + (yy**2)/(2*sy_mm**2) + (zz**2)/(2*sz_mm**2))
    gaussian = np.exp(exponent)
    
    # Rotación aleatoria (20-90 grados)
    angle = np.random.uniform(20, 90)
    gaussian = rotate(gaussian, angle, axes=(0,1), reshape=False, order=1)
    
    # Degradación a resolución ADNI (1x1x4)
    # Factor de reducción: voxel_size / res
    factors = (int(voxel_size[0]/res), int(voxel_size[1]/res), int(voxel_size[2]/res))
    low_res_blob = downscale_local_mean(gaussian, factors)
    
    if low_res_blob.max() > 0:
        low_res_blob /= low_res_blob.max()
        
    return low_res_blob * strength

# --- BUCLE PILOTO (10 sujetos) ---
# (Implementación simplificada para validación visual)
subjects = [f for f in os.listdir(INPUT_DIR) if f.endswith(".nii.gz")][:10]

for filename in subjects:
    nii = nib.load(os.path.join(INPUT_DIR, filename))
    data = nii.get_fdata().astype(np.float32)
    aff, hdr = nii.affine, nii.header
    pixdim = hdr.get_zooms()[:3]
    
    # Máscara de seguridad (Multi-Otsu)
    thresh = threshold_multiotsu(data[data > 0], classes=3)
    brain_mask = data > thresh[0] # Tejido cerebral
    valid_coords = np.argwhere(brain_mask)
    
    # Insertar 5 microhemorragias por sujeto para la prueba
    final_label = np.zeros(data.shape, dtype=np.uint8)
    for _ in range(5):
        idx = np.random.randint(len(valid_coords))
        cx, cy, cz = valid_coords[idx]
        
        # Volumen basado en tus pseudomáscaras logradas (aprox 2-8 mm3)
        vol = np.random.uniform(2, 8)
        blob = generate_sCMB_momeni(vol, pixdim)
        
        # Insertar blob en la imagen
        # (Cálculo de offsets para centrar el patrón)
        dx, dy, dz = [s // 2 for s in blob.shape]
        x_s, x_e = cx-dx, cx-dx+blob.shape[0]
        y_s, y_e = cy-dy, cy-dy+blob.shape[1]
        z_s, z_e = cz-dz, cz-dz+blob.shape[2]
        
        # Control de bordes
        if x_s >= 0 and x_e < data.shape[0] and y_s >= 0 and y_e < data.shape[1] and z_s >= 0 and z_e < data.shape[2]:
            roi = data[x_s:x_e, y_s:y_e, z_s:z_e]
            data[x_s:x_e, y_s:y_e, z_s:z_e] = roi * (1 - blob)
            final_label[x_s:x_e, y_s:y_e, z_s:z_e] = np.maximum(final_label[x_s:x_e, y_s:y_e, z_s:z_e], (blob > 0.5))

    # Guardar resultados
    nib.save(nib.Nifti1Image(data, aff, hdr), os.path.join(OUTPUT_DIR, "imagesTr", filename))
    nib.save(nib.Nifti1Image(final_label, aff, hdr), os.path.join(OUTPUT_DIR, "labelsTr", filename))
    print(f"Sujeto {filename} procesado.")